In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 37.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=9297a681fff5b1a15e46ab739fd5b70a766996b70bc55f1efbaeaa799ed4175f
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# ─────────────────────────────────────────────────────────────
# BB84 Quantum Key Distribution — With Attacker (Eve)
# Agents: Alice (sender) | Eve (attacker) | Bob (receiver)
#
# Eve's strategy: intercept-and-resend
#   She measures each qubit in a randomly chosen basis, then
#   re-prepares and forwards a new qubit to Bob.
#   When she guesses the wrong basis (~50% of the time), she
#   disturbs the state — introducing a ~25% QBER overall,
#   well above the 11% detection threshold.
# ─────────────────────────────────────────────────────────────

simulator = BasicSimulator()


# ── Quantum Random Number Generator ───────────────────────────────────────────
# Batches of 24 to stay within BasicSimulator's 24-qubit hard limit.
# Prepares qubits in |+⟩ = (|0⟩+|1⟩)/√2 — pure quantum randomness,
# no Python random module used anywhere.

def quantum_random_bits(n):
    bits  = []
    batch = 20
    while len(bits) < n:
        size = min(batch, n - len(bits))
        qc   = QuantumCircuit(size, size)
        qc.h(range(size))
        qc.measure(range(size), range(size))
        job    = simulator.run(transpile(qc, simulator), shots=1)
        result = list(job.result().get_counts().keys())[0]
        bits  += [int(b) for b in reversed(result)]
    return bits[:n]


# ══════════════════════════════════════════════════════════════
# ALICE — prepares and sends qubits
# ══════════════════════════════════════════════════════════════

def alice_prepare(n):
    """
    Alice generates n random bits and n random bases, then encodes
    each bit into a single-qubit circuit:

      bit=0, basis=0 (rectilinear) → |0⟩   (do nothing)
      bit=1, basis=0 (rectilinear) → |1⟩   (X gate)
      bit=0, basis=1 (diagonal)   → |+⟩   (H gate)
      bit=1, basis=1 (diagonal)   → |−⟩   (X then H)

    She sends the circuits to Bob (via Eve, unknown to her).
    """
    bits  = quantum_random_bits(n)
    bases = quantum_random_bits(n)

    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)       # encode bit value
        if basis == 1:
            qc.h(0)       # rotate to diagonal basis
        circuits.append(qc)

    print("[Alice] bits:  ", bits)
    print("[Alice] bases: ", bases, "  (0=rectilinear, 1=diagonal)")
    return circuits, bits, bases


# ══════════════════════════════════════════════════════════════
# EVE — intercepts, measures, re-sends (intercept-and-resend)
# ══════════════════════════════════════════════════════════════

def eve_intercept(circuits):
    """
    Eve sits between Alice and Bob on the quantum channel.
    For each qubit she:

      1. Picks a random basis (quantum RNG — she has no idea what
         Alice used, so she guesses correctly only ~50% of the time).
      2. Measures — this irrevocably collapses the quantum state
         (no-cloning theorem means she cannot copy it first).
      3. Re-prepares a NEW qubit encoding her measurement result
         in her chosen basis and forwards it to Bob.

    Consequence: when Eve's basis matches Alice's, Bob gets the
    correct bit. When it doesn't, Eve sends a disturbed state,
    causing ~25% errors in the final sifted key.
    """
    eve_bases   = quantum_random_bits(len(circuits))
    eve_results = []
    forwarded   = []          # circuits Eve sends on to Bob

    for qc, basis in zip(circuits, eve_bases):

        # ── Step 1 & 2: Eve measures in her chosen basis ──────────
        eve_qc = QuantumCircuit(1, 1)
        eve_qc.compose(qc, inplace=True)   # apply Alice's encoding
        if basis == 1:
            eve_qc.h(0)                    # rotate to Eve's basis
        eve_qc.measure(0, 0)

        job = simulator.run(transpile(eve_qc, simulator), shots=1)
        measured_bit = int(list(job.result().get_counts().keys())[0])
        eve_results.append(measured_bit)

        # ── Step 3: Eve re-prepares and forwards to Bob ───────────
        new_qc = QuantumCircuit(1, 1)
        if measured_bit == 1:
            new_qc.x(0)
        if basis == 1:
            new_qc.h(0)
        forwarded.append(new_qc)

    print("[Eve]   bases:   ", eve_bases)
    print("[Eve]   results: ", eve_results)
    print(f"[Eve]   intercepted all {len(circuits)} qubits — Bob receives Eve's re-sent qubits")
    return forwarded, eve_bases, eve_results


# ══════════════════════════════════════════════════════════════
# BOB — receives and measures qubits (from Eve, not Alice)
# ══════════════════════════════════════════════════════════════

def bob_measure(circuits, n):
    """
    Bob independently chooses n random bases and measures each qubit.
    He is unaware the qubits were intercepted and re-sent by Eve.
    """
    bases   = quantum_random_bits(n)
    results = []

    for qc, basis in zip(circuits, bases):
        full_qc = QuantumCircuit(1, 1)
        full_qc.compose(qc, inplace=True)
        if basis == 1:
            full_qc.h(0)      # rotate from diagonal basis before measuring
        full_qc.measure(0, 0)

        job = simulator.run(transpile(full_qc, simulator), shots=1)
        results.append(int(list(job.result().get_counts().keys())[0]))

    print("[Bob]   bases:   ", bases)
    print("[Bob]   results: ", results)
    return results, bases


# ══════════════════════════════════════════════════════════════
# SIFTING — classical public channel (Alice ↔ Bob)
# ══════════════════════════════════════════════════════════════

def sift_key(alice_bits, alice_bases, bob_results, bob_bases):
    """
    Alice and Bob announce their bases publicly (NOT their bits).
    They keep only positions where both chose the same basis.
    Eve cannot prevent this — it happens over a classical channel.
    """
    alice_key, bob_key, kept = [], [], []

    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_key.append(alice_bits[i])
            bob_key.append(bob_results[i])
            kept.append(i)

    pct = len(alice_key) / len(alice_bits) * 100
    print(f"\n[Sift]  Kept {len(alice_key)}/{len(alice_bits)} bits ({pct:.0f}%)")
    print("[Alice sifted key]", alice_key)
    print("[Bob   sifted key]", bob_key)
    return alice_key, bob_key


# ══════════════════════════════════════════════════════════════
# ERROR RATE CHECK — attack detection
# ══════════════════════════════════════════════════════════════

def check_error_rate(alice_key, bob_key, sample_fraction=0.2, threshold=0.11):
    """
    Alice and Bob sacrifice a sample of their sifted key bits by
    revealing them over the classical channel to compute the QBER.

    No attacker → QBER ≈ 0%
    Eve intercepts all qubits → QBER ≈ 25%

    The standard security threshold is 11%. Above this, the protocol
    aborts and the key is discarded.
    """
    n_sample = max(1, math.ceil(len(alice_key) * sample_fraction))

    # Use quantum RNG to pick which positions to sample
    rand_bits = quantum_random_bits(len(alice_key))
    indices   = [i for i, b in enumerate(rand_bits) if b == 1][:n_sample]
    if not indices:
        indices = [0]

    errors = sum(alice_key[i] != bob_key[i] for i in indices)
    qber   = errors / len(indices)

    print(f"\n[Check] Sample size: {len(indices)}, Errors: {errors}")
    print(f"[Check] QBER: {qber:.2%}  (security threshold: {threshold:.0%})")

    if qber > threshold:
        print("[Check] ⚠️  QBER exceeds threshold — Eve detected! Key discarded.")
        return qber, True
    else:
        print("[Check] ✅ QBER within safe range — attack not detected this run.")
        return qber, False


# ══════════════════════════════════════════════════════════════
# RUN THE PROTOCOL
# ══════════════════════════════════════════════════════════════

N = 100   # number of qubits Alice sends

print("=" * 56)
print("BB84 — With attacker (Eve: intercept-and-resend)")
print("=" * 56)

# Step 1 — Alice encodes her qubits
circuits, alice_bits, alice_bases = alice_prepare(N)

# Step 2 — Eve intercepts every qubit, measures, and re-sends
print()
forwarded, eve_bases, eve_results = eve_intercept(circuits)

# Step 3 — Bob measures Eve's re-sent qubits (unaware of interception)
print()
bob_results, bob_bases = bob_measure(forwarded, N)

# Step 4 — Alice and Bob sift on the classical channel
alice_key, bob_key = sift_key(alice_bits, alice_bases, bob_results, bob_bases)

# Step 5 — Check error rate and decide whether to trust the key
qber, attack_detected = check_error_rate(alice_key, bob_key)

# Step 6 — Summary
print("\n── Summary " + "─" * 45)
print(f"  Qubits sent:         {N}")
print(f"  Sifted key bits:     {len(alice_key)}")
print(f"  QBER:                {qber:.2%}  (expected ~25% with Eve intercepting all qubits)")
print(f"  Attack detected:     {'YES ⚠️' if attack_detected else 'NO  (try larger N for reliability)'}")
if attack_detected:
    print("\n  Protocol aborted — shared key is insecure and discarded.")
else:
    print("\n  Key accepted (Eve got lucky this run — increase N to reduce this risk).")

BB84 — With attacker (Eve: intercept-and-resend)
[Alice] bits:   [1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]
[Alice] bases:  [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0]   (0=rectilinear, 1=diagonal)

[Eve]   bases:    [1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 